In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import ggblab

In [3]:
%ggblab api getVersion()

[ggblab] created GeoGebra singleton and stored as 'ggb' in user namespace


'5.2.909.9'

In [4]:
import polars as pl

In [5]:
from ggblab_extra.sympy import attach_object2d, attach_object3d, enumerate_plane_members

In [6]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [10]:
%%ggb
Circle((0,0), 1)
Line((-1,-1), (1,2))

['c', 'f']

In [11]:
df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)

In [21]:
df2 = attach_object2d(df)

In [22]:
df2["Name", "Command", "Value", "object2d"]

Name,Command,Value,object2d
str,str,str,object
"""c""","""Circle((0, 0), 1)""","""c: x² + y² = 1""","Object2D(kind='circle', obj=Circle(Point2D(0, 0), 1), value='c: x² + y² = 1', command='Circle((0, 0), 1)')"
"""f""","""Line((-1, -1), (1, 2))""","""f: -3x + 2y = 1""","Object2D(kind='line', obj=Line2D(Point2D(0, 1/2), Point2D(-2, -5/2)), value='f: -3x + 2y = 1', command='Line((-1, -1), (1, 2))')"


In [23]:
pl.Config.set_fmt_str_lengths(1000)
print(str(df2["Name", "Command", "Value", "object2d"]))

shape: (2, 4)
┌──────┬────────────────────────┬─────────────────┬────────────────────────────────────────────────┐
│ Name ┆ Command                ┆ Value           ┆ object2d                                       │
│ ---  ┆ ---                    ┆ ---             ┆ ---                                            │
│ str  ┆ str                    ┆ str             ┆ object                                         │
╞══════╪════════════════════════╪═════════════════╪════════════════════════════════════════════════╡
│ c    ┆ Circle((0, 0), 1)      ┆ c: x² + y² = 1  ┆ Object2D(kind='circle', obj=Circle(Point2D(0,  │
│      ┆                        ┆                 ┆ 0), 1), value='c: x² + y² = 1',                │
│      ┆                        ┆                 ┆ command='Circle((0, 0), 1)')                   │
│ f    ┆ Line((-1, -1), (1, 2)) ┆ f: -3x + 2y = 1 ┆ Object2D(kind='line', obj=Line2D(Point2D(0,    │
│      ┆                        ┆                 ┆ 1/2), Point2D(-2, -5/2)),

In [24]:
o1 = df2.filter(pl.col("Name") == "c")["object2d"].item().obj

In [25]:
o2 = df2.filter(pl.col("Name") == "f")["object2d"].item().obj

In [26]:
o1, o2

(Circle(Point2D(0, 0), 1), Line2D(Point2D(0, 1/2), Point2D(-2, -5/2)))

In [28]:
from sympy import nonlinsolve, linsolve, solve, Eq, discriminant
from sympy import symbols
x, y, z = symbols('x y z')

In [29]:
p1, p2 = (o1.equation(x=x, y=y), o2.equation(x=x, y=y))

In [30]:
p1, p2

(x**2 + y**2 - 1, 3*x - 2*y + 1)

In [31]:
p2.subs({x: 3, y:4})

2

In [32]:
s = solve(Eq(p2, 0), y)
s

[3*x/2 + 1/2]

In [33]:
discriminant(p1.subs({y: s[0]})).evalf()

12.0000000000000

In [34]:
p1.subs({y: s[0]})

x**2 + (3*x/2 + 1/2)**2 - 1

In [35]:
s2 = nonlinsolve([p1.subs({y: s[0]})], [x])
s2

{(-3/13 + 4*sqrt(3)/13,), (-4*sqrt(3)/13 - 3/13,)}

In [36]:
# [(e[0].evalf(), e[1].evalf()) 
[e[0].evalf() for e in s2]

[0.302169479251962, -0.763707940790424]

In [37]:
o1.intersect(o2).evalf()

{Point2D(-0.763707940790424, -0.645561911185636), Point2D(0.302169479251962, 0.953254218877943)}

In [38]:
s2 = nonlinsolve([p1, p2], [x, y])

In [39]:
[(e[0].evalf(), e[1].evalf()) for e in s2]

[(0.302169479251962, 0.953254218877943),
 (-0.763707940790424, -0.645561911185636)]